<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">چسباندن خروجی <bdi dir="ltr">Head</bdi>ها و شمارش جدول <bdi dir="ltr">Attention</bdi></h1>
<p style="text-align:right">درس 42 از 92 · ادغام سرها و هزینهٔ زمینهٔ بلند · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">36-merge-heads</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-01/36-merge-heads.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">خروجی واقعی <bdi dir="ltr">Head</bdi>ها را ادغام کنید و <bdi dir="ltr">Projection</bdi> نهایی را با ماژول پروژه تطبیق دهید.</p><p style="text-align:right"><span class="phrase-lead" style="white-space:nowrap">پیش‌نیاز: تقسیم</span> <bdi dir="ltr">Head</bdi> و شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">weights @ V</code> را بشناسید.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۴۵–۸۰ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">attended</code> شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(2,3,5,4)</code> دارد، خروجی ادغام چه شکلی است؟ چرا <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">reshape</code> مستقیم ممکن است ترتیب موقعیت‌ها را مخلوط کند؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.config import ModelConfig
from mini_gpt.attention import CausalSelfAttention
attention = CausalSelfAttention(ModelConfig(12,8,12,3,1,0.)).eval()
x = torch.randn(2,5,12)
trace = {}
with torch.no_grad():
    output = attention(x,trace=trace)
attended = trace['weighted_values']
print('per-head output:',attended.shape)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">merge_heads(attended)</code> را بنویسید: <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,H,T,D)</code> به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,T,H*D)</code>. هر زمان باید ویژگی‌های <bdi dir="ltr">Head</bdi>های همان زمان را کنار هم داشته باشد.</p>
</div>

In [ ]:
def merge_heads(attended):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = merge_heads(attended)
    if result is None: return False
    torch.testing.assert_close(attention.output(result),output)
    for B,H,T,D in ((1,2,3,4),(2,3,2,1)):
        a = torch.arange(B*H*T*D).reshape(B,H,T,D)
        merged = merge_heads(a)
        assert merged.shape == (B,T,H*D)
        for h in range(H):
            assert torch.equal(merged[:,:,h*D:(h+1)*D],a[:,h])
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">T</code> را دو برابر کنید و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">B=2</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">H=3</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">float32</code> را ثابت بگذارید. فقط حافظهٔ یک جدول وزن را حساب می‌کنیم، نه کل حافظهٔ مدل.</p>
</div>

In [ ]:
for T in (16,32,64):
    elements = 2*3*T*T
    print('T, elements, KiB:',T,elements,elements*4/1024)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">contiguous</code> نمی‌تواند محور معنایی اشتباه را اصلاح کند. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">repair_merge(a)</code> را اصلاح کنید؛ دادهٔ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">arange</code> کمک می‌کند محل اشتباه را ببینید.</p>
</div>

In [ ]:
marker = torch.arange(24).reshape(1,2,3,4)
print('wrong first position:',marker.contiguous().view(1,3,8)[0,0])
print('two correct head pieces:',marker[0,0,0],marker[0,1,0])

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def repair_merge(a):
    # TODO
    return None

In [ ]:
def test_repair():
    result = repair_merge(marker)
    if result is None: return False
    assert torch.equal(result[0,0],torch.cat((marker[0,0,0],marker[0,1,0])))
    a = torch.arange(60).reshape(2,3,5,2)
    assert torch.equal(repair_merge(a)[:,:,2:4],a[:,1])
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">مقایسه با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">attention.output</code> روی همان وزن‌ها انجام شد؛ <bdi dir="ltr">Dropout</bdi> صفر است. این بخش انتهای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">CausalSelfAttention</code> مشترک میان <bdi dir="ltr">v4</bdi> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code> نهایی است.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چرا دوبرابرکردن <bdi dir="ltr">Batch</bdi> و دوبرابرکردن <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">T</code> اثر یکسانی بر حافظهٔ جدول وزن ندارند؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-01/36-merge-heads.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/36-merge-heads.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>